# F-Chat Training Pipeline

Two-stage curriculum fine-tuning of `meta-llama/Llama-2-7b-hf` for medical
symptom assessment and structured clinical reporting.

| Phase | Goal | Data |
|---|---|---|
| 1 | Raw medical model (domain-adaptive pretraining) | RedPajama, PubMed abstracts (~16M), EPFL clinical guidelines (~36k) |
| 2 | F-Chat (dialogue alignment) | ~100k doctor-patient Q&A pairs |

Both phases use LoRA. The final checkpoint is merged and quantized to 4-bit
GPTQ for deployment.

## 1. Environment setup

In [ ]:
!pip install -q -r ../requirements.txt

In [ ]:
import sys
sys.path.insert(0, "../src")

import fchat
from fchat import config as cfg

cfg.DEV_MODE = True
cfg.ensure_directories()

print("DEV_MODE:", cfg.DEV_MODE)
print("MODE:", cfg.MODE)
print("BASE_MODEL:", cfg.BASE_MODEL)

## 2. Tokenizer

Loads the F-Chat tokenizer (local export if present, otherwise from
`BASE_MODEL`).

In [ ]:
tokenizer = fchat.load_tokenizer()

## 3. Phase 1 data preparation

Streams and merges RedPajama, PubMed and EPFL clinical guidelines into
`data/phase1_medical_base.jsonl`. Set `cfg.DEV_MODE = False` for a full-scale
build.

In [ ]:
phase1_jsonl = fchat.build_phase1_jsonl(
    out_path=cfg.PHASE1_JSONL,
    dev_mode=cfg.DEV_MODE,
)

In [ ]:
phase1_tokenized = fchat.tokenize_jsonl(phase1_jsonl, tokenizer)
phase1_tokenized

## 4. Phase 1 training: raw medical model

In [ ]:
phase1_trainer = fchat.run_training_phase(
    phase_name="PHASE1",
    base_model_path=cfg.BASE_MODEL,
    tokenized_dataset=phase1_tokenized,
    tokenizer=tokenizer,
    checkpoint_dir=cfg.PHASE1_CHECKPOINT_DIR,
    output_dir=cfg.PHASE1_OUTPUT_DIR,
    training_args=cfg.PHASE1_TRAINING_ARGS,
    mode=cfg.MODE,
)

In [ ]:
print(fchat.generate_sample(
    phase1_trainer.model,
    tokenizer,
    prompt="A 45-year-old patient presents with persistent cough and fever.",
))

## 5. Phase 2 data preparation

Builds `data/phase2_medical_qa.jsonl` from the local Medical QA CSV
(`cfg.PHASE2_QA_CSV`). Download the Kaggle dataset and place it at that path
before running this cell with `cfg.DEV_MODE = False`.

In [ ]:
phase2_jsonl = fchat.build_phase2_jsonl(
    out_path=cfg.PHASE2_JSONL,
    csv_path=cfg.PHASE2_QA_CSV,
    dev_mode=cfg.DEV_MODE,
)

In [ ]:
phase2_tokenized = fchat.tokenize_jsonl(phase2_jsonl, tokenizer)
phase2_tokenized

## 6. Phase 2 training: F-Chat dialogue alignment

In [ ]:
phase2_trainer = fchat.run_training_phase(
    phase_name="PHASE2",
    base_model_path=cfg.PHASE1_OUTPUT_DIR,
    tokenized_dataset=phase2_tokenized,
    tokenizer=tokenizer,
    checkpoint_dir=cfg.CHECKPOINT_DIR,
    output_dir=cfg.OUTPUT_DIR,
    training_args=cfg.PHASE2_TRAINING_ARGS,
    mode=cfg.MODE,
)

## 7. Evaluation

In [ ]:
eval_prompts = [
    "Q: I have had a headache and blurred vision for two days. A:",
    "Q: What are the first-line treatments for type 2 diabetes? A:",
    "Q: My child has a fever of 39C and a rash. What should I do? A:",
]

for prompt in eval_prompts:
    print("PROMPT:", prompt)
    print(fchat.generate_sample(phase2_trainer.model, tokenizer, prompt))
    print("-" * 80)

## 8. GPTQ quantization

Builds calibration examples from both phase corpora and quantizes the
merged Phase 2 model to 4-bit GPTQ. Requires `auto-gptq` to be installed.

In [ ]:
calibration_examples = fchat.build_calibration_examples(
    tokenizer,
    jsonl_paths=[cfg.PHASE1_JSONL, cfg.PHASE2_JSONL],
    n_samples=cfg.N_CALIBRATION_SAMPLES,
)

In [ ]:
quantization_succeeded = fchat.quantize_to_gptq(
    model_dir=cfg.OUTPUT_DIR,
    quant_dir=cfg.QUANT_DIR,
    tokenizer=tokenizer,
    calibration_examples=calibration_examples,
)

print("Quantization succeeded:", quantization_succeeded)

## 9. Model card and export

In [ ]:
card_target_dir = cfg.QUANT_DIR if quantization_succeeded else cfg.OUTPUT_DIR
card_path = fchat.write_model_card(card_target_dir)
print("Model card written to:", card_path)

In [ ]:
import os

target_dir = cfg.QUANT_DIR if quantization_succeeded else cfg.OUTPUT_DIR
print(f"Final artifacts in: {target_dir}")
for name in sorted(os.listdir(target_dir)):
    print(" -", name)